---

_You are currently looking at **version 1.2** of this notebook. To download notebooks and datafiles, as well as get help on Jupyter notebooks in the Coursera platform, visit the [Jupyter Notebook FAQ](https://www.coursera.org/learn/python-social-network-analysis/resources/yPcBs) course resource._

---

# Assignment 4

In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle

---

## Part 1 - Random Graph Identification

For the first part of this assignment you will analyze randomly generated graphs and determine which algorithm created them.

In [2]:
P1_Graphs = pickle.load(open('A4_graphs','rb'))
P1_Graphs

<br>
`P1_Graphs` is a list containing 5 networkx graphs. Each of these graphs were generated by one of three possible algorithms:
* Preferential Attachment (`'PA'`)
* Small World with low probability of rewiring (`'SW_L'`)
* Small World with high probability of rewiring (`'SW_H'`)

Anaylze each of the 5 graphs and determine which of the three algorithms generated the graph.

*The `graph_identification` function should return a list of length 5 where each element in the list is either `'PA'`, `'SW_L'`, or `'SW_H'`.*

In [12]:
def graph_identification():
#     import matplotlib.pyplot as plt

#     for G in P1_Graphs:        
#         degrees = G.degree()
#         degree_values = sorted(set(degrees.values()))
#         histogram = [list(degrees.values()).count(i) / float(nx.number_of_nodes(G)) for i in degree_values]
    
#         plt.plot(degree_values, histogram, 'o')
#         plt.xlabel('Degree')
#         plt.ylabel('Fraction of Nodes')
#         plt.xscale('log')
#         plt.yscale('log')
#         plt.show()
        
# for Small World: lower "p" - higher average_shortest_path_length and average_clustering
#         print(nx.average_shortest_path_length(G)) 
#         print(nx.average_clustering(G))
        
    return ['PA', 'SW_L', 'SW_L', 'PA', 'SW_H']  # Your Answer Here

graph_identification()

['PA', 'SW_L', 'SW_L', 'PA', 'SW_H']

---

## Part 2 - Company Emails

For the second part of this assignment you will be workking with a company's email network where each node corresponds to a person at the company, and each edge indicates that at least one email has been sent between two people.

The network also contains the node attributes `Department` and `ManagementSalary`.

`Department` indicates the department in the company which the person belongs to, and `ManagementSalary` indicates whether that person is receiving a management position salary.

In [4]:
G = nx.read_gpickle('email_prediction.txt')

print(nx.info(G))

Name: 
Type: Graph
Number of nodes: 1005
Number of edges: 16706
Average degree:  33.2458


### Part 2A - Salary Prediction

Using network `G`, identify the people in the network with missing values for the node attribute `ManagementSalary` and predict whether or not these individuals are receiving a management position salary.

To accomplish this, you will need to create a matrix of node features using networkx, train a sklearn classifier on nodes that have `ManagementSalary` data, and predict a probability of the node receiving a management salary for nodes where `ManagementSalary` is missing.



Your predictions will need to be given as the probability that the corresponding employee is receiving a management position salary.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.88 or higher will receive full points, and with an AUC of 0.82 or higher will pass (get 80% of the full points).

Using your trained classifier, return a series of length 252 with the data being the probability of receiving management salary, and the index being the node id.

    Example:
    
        1       1.0
        2       0.0
        5       0.8
        8       1.0
            ...
        996     0.7
        1000    0.5
        1001    0.0
        Length: 252, dtype: float64

In [5]:
def salary_predictions():
    
    # Your Code Here
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import roc_curve, auc
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import cross_val_score

    # загружаем в DataFrame узлы графа и их атрибуты
    df = pd.DataFrame(index=G.nodes())
    df['Department'] = pd.Series(nx.get_node_attributes(G, 'Department'))
    df['ManagementSalary'] = pd.Series(nx.get_node_attributes(G, 'ManagementSalary'))
    
    # добавляем в качестве свойств базовые функции кластеризации
    df['clustering'] = pd.Series(nx.clustering(G))
    df['degree'] = pd.Series(G.degree())
    df['triangles'] = pd.Series(nx.triangles(G))
    df['square_clustering'] = pd.Series(nx.square_clustering(G))
    
    # отбираем записи с определённым ManagementSalary (на них будет тренироваться модель)
    df_valid = df.dropna(axis=0, how='any', subset=['ManagementSalary'])  
    # и записи с неопределённым ManagementSalary (для них будет делаться предсказание)
    df_nan = df[~df.index.isin(df_valid.index)]
    df_nan.drop('ManagementSalary', axis = 1, inplace=True)
    
    # формируем массив свойств и массив значения (ManagementSalary)
    X = df_valid.drop('ManagementSalary', axis = 1)
    y = df_valid['ManagementSalary']
    
    # оценка модели на тренировочных и тестовых данных
#     X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
#     clf = GradientBoostingClassifier(learning_rate = 0.01, max_depth = 3, random_state = 0).fit(X_train, y_train)
#     y_score = clf.decision_function(X_test)
#     fpr_svm, tpr_svm, _ = roc_curve(y_test, y_score)
#     roc_auc_svm = auc(fpr_svm, tpr_svm)
#     print("AUC = {:.2f}".format(roc_auc_svm))
#     print('Cross-validation (AUC)', cross_val_score(clf, X, y, cv=5, scoring = 'roc_auc'))

    # обучение модели на GradientBoostingClassifier и получение предсказаний
    clf = GradientBoostingClassifier(learning_rate = 0.01, max_depth = 3, random_state = 0).fit(X, y)
    y_predict_proba = clf.predict_proba(df_nan)
    y_resut_series = pd.Series(y_predict_proba[:,1], index=df_nan.index)
    
    return y_resut_series # Your Answer Here

salary_predictions()

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


1       0.132156
2       0.321811
5       0.679970
8       0.272484
14      0.132578
18      0.127697
27      0.128677
30      0.426505
31      0.391298
34      0.219416
37      0.215048
40      0.127697
45      0.127937
54      0.319828
55      0.377534
60      0.391298
62      0.679970
65      0.435128
77      0.074548
79      0.131505
97      0.074548
101     0.074548
103     0.391298
108     0.131505
113     0.374711
122     0.072202
141     0.128193
142     0.679970
144     0.077184
145     0.433169
          ...   
913     0.072202
914     0.074548
915     0.070878
918     0.160843
923     0.104859
926     0.104859
931     0.070878
934     0.077184
939     0.070878
944     0.070878
945     0.104859
947     0.108820
950     0.081152
951     0.072588
953     0.070878
959     0.070878
962     0.070878
963     0.272484
968     0.108820
969     0.108820
974     0.104859
984     0.077184
987     0.104859
989     0.108820
991     0.118073
992     0.070878
994     0.070878
996     0.0708

### Part 2B - New Connections Prediction

For the last part of this assignment, you will predict future connections between employees of the network. The future connections information has been loaded into the variable `future_connections`. The index is a tuple indicating a pair of nodes that currently do not have a connection, and the `Future Connection` column indicates if an edge between those two nodes will exist in the future, where a value of 1.0 indicates a future connection.

In [6]:
future_connections = pd.read_csv('Future_Connections.csv', index_col=0, converters={0: eval})
future_connections.head(10)

,Future Connection
"(6, 840)",0.0
"(4, 197)",0.0
"(620, 979)",0.0
"(519, 872)",0.0
"(382, 423)",0.0
"(97, 226)",1.0
"(349, 905)",0.0
"(429, 860)",0.0
"(309, 989)",0.0
"(468, 880)",0.0


Using network `G` and `future_connections`, identify the edges in `future_connections` with missing values and predict whether or not these edges will have a future connection.

To accomplish this, you will need to create a matrix of features for the edges found in `future_connections` using networkx, train a sklearn classifier on those edges in `future_connections` that have `Future Connection` data, and predict a probability of the edge being a future connection for those edges in `future_connections` where `Future Connection` is missing.



Your predictions will need to be given as the probability of the corresponding edge being a future connection.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.88 or higher will receive full points, and with an AUC of 0.82 or higher will pass (get 80% of the full points).

Using your trained classifier, return a series of length 122112 with the data being the probability of the edge being a future connection, and the index being the edge as represented by a tuple of nodes.

    Example:
    
        (107, 348)    0.35
        (542, 751)    0.40
        (20, 426)     0.55
        (50, 989)     0.35
                  ...
        (939, 940)    0.15
        (555, 905)    0.35
        (75, 101)     0.65
        Length: 122112, dtype: float64

In [7]:
def new_connections_predictions():
    
    # Your Code Here
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import roc_curve, auc
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import cross_val_score

    # добавляем в качестве свойств базовые функции рёбер
    future_connections['preferential attachment'] = [i[2] for i in nx.preferential_attachment(G, future_connections.index)]
    future_connections['Common Neighbors'] = future_connections.index.map(lambda node: len(list(nx.common_neighbors(G, node[0], node[1]))))
    
    # отбираем записи с определённым Future Connection (на них будет тренироваться модель)
    future_valid = future_connections.dropna(axis=0, how='any', subset=['Future Connection'])  
    # и записи с неопределённым Future Connection (для них будет делаться предсказание)
    future_nan = future_connections[~future_connections.index.isin(future_valid.index)]   
    future_nan.drop('Future Connection', axis = 1, inplace=True)
    
    # формируем массив свойств и массив значения (Future Connection)
    X = future_valid.drop('Future Connection', axis = 1)
    y = future_valid['Future Connection']

    # оценка модели на тренировочных и тестовых данных
#     X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
#     clf = GradientBoostingClassifier(learning_rate = 0.01, max_depth = 3, random_state = 0).fit(X_train, y_train)
#     y_score = clf.decision_function(X_test)
#     fpr_svm, tpr_svm, _ = roc_curve(y_test, y_score)
#     roc_auc_svm = auc(fpr_svm, tpr_svm)
#     print("AUC = {:.2f}".format(roc_auc_svm))
#     print('Cross-validation (AUC)', cross_val_score(clf, X, y, cv=5, scoring = 'roc_auc'))
    
    # обучение модели на GradientBoostingClassifier и получение предсказаний
    clf = GradientBoostingClassifier(learning_rate = 0.01, max_depth = 3, random_state = 0).fit(X, y)
    y_predict_proba = clf.predict_proba(future_nan)
    y_resut_series = pd.Series(y_predict_proba[:,1], index=future_nan.index)
    
    return y_resut_series # Your Answer Here

new_connections_predictions()

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


(107, 348)    0.051404
(542, 751)    0.038817
(20, 426)     0.385734
(50, 989)     0.038817
(942, 986)    0.038817
(324, 857)    0.038817
(13, 710)     0.133634
(19, 271)     0.133634
(319, 878)    0.038817
(659, 707)    0.038817
(49, 843)     0.038817
(208, 893)    0.038817
(377, 469)    0.038817
(405, 999)    0.039255
(129, 740)    0.039255
(292, 618)    0.039255
(239, 689)    0.038817
(359, 373)    0.038817
(53, 523)     0.051404
(276, 984)    0.038817
(202, 997)    0.038817
(604, 619)    0.051404
(270, 911)    0.038817
(261, 481)    0.093490
(200, 450)    0.510417
(213, 634)    0.038817
(644, 735)    0.051404
(346, 553)    0.038817
(521, 738)    0.038817
(422, 953)    0.039255
                ...   
(672, 848)    0.038817
(28, 127)     0.646928
(202, 661)    0.038817
(54, 195)     0.656348
(295, 864)    0.038817
(814, 936)    0.038817
(839, 874)    0.038817
(139, 843)    0.038817
(461, 544)    0.038817
(68, 487)     0.038817
(622, 932)    0.038817
(504, 936)    0.039255
(479, 528) 